# LLM Query Writing Strategies for Search Retrieval

## Experiment Summary

This notebook evaluates different LLM-based query writing strategies for improving search retrieval performance on the BRIGHT biology dataset (103 queries). The experiment compares several approaches:

### Strategies Evaluated

1. **Baseline (Original Queries)**: Direct use of user queries without modification
2. **Query Writing**: Simple LLM-based query rewriting/optimization
3. **HyDE (Hypothetical Document Embeddings)**: Generating hypothetical documents that capture relevance patterns
4. **PRF (Pseudo-Relevance Feedback)**: Query refinement based on initial search results
5. **Multi-Query Generation**: Generating multiple diverse queries and aggregating results

### Key Results

**Baseline Performance:**
- Success @ 1: 15.5% (16/103)
- Success @ 5: 32.0% (33/103)
- Success @ 10: 40.8% (42/103)

**Query Writing:**
- Success @ 1: 41.7% (43/103) - **+26.2% improvement**
- Success @ 5: 61.2% (63/103) - **+29.2% improvement**
- Success @ 10: 68.0% (70/103) - **+27.2% improvement**

**HyDE (Hypothetical Document Embeddings):**
- Success @ 1: 49.5% (51/103) - **+34.0% improvement**
- Success @ 5: 68.9% (71/103) - **+36.9% improvement**
- Success @ 10: 73.8% (76/103) - **+33.0% improvement**

**PRF (Pseudo-Relevance Feedback):**
- Success @ 1: 42.7% (44/103) - **+27.2% improvement**
- Success @ 5: 66.0% (68/103) - **+34.0% improvement**
- Success @ 10: 69.9% (72/103) - **+29.1% improvement**

**Multi-Query Generation:**
- Success @ 1: 23.3% (24/103) - **+7.8% improvement**
- Success @ 5: 42.7% (44/103) - **+10.7% improvement**
- Success @ 20: 74.8% (77/103) - **+34.0% improvement**

### Findings

- **All LLM-based strategies significantly outperform the baseline**, with improvements ranging from 7.8-37 percentage points
- **HyDE performs best overall**, achieving nearly 50% success at K=1 and 74% at K=20
- **PRF shows strong performance** (66% at K=5, 70% at K=20), leveraging corpus-specific terminology from initial results
- **Multi-query generation excels at higher K values** (75% at K=20), demonstrating the power of query diversity for recall
- **Query writing provides substantial gains** with a simpler approach than HyDE or PRF

### Methodology

- Dataset: BRIGHT biology examples (103 queries)
- Search: Hybrid search using Weaviate
- Evaluation: Success@K metric (whether any gold document appears in top-K results)
- Model: OpenAI GPT-5-mini via DSPy

In [ ]:
from datasets import load_dataset

queries = load_dataset("xlangai/BRIGHT", "examples")["biology"]

In [13]:
queries[0]

{'query': 'Claim in article about why insects are attracted to light\nIn this article they are addressing the reason insects are attracted to light when they say\nHeat radiation as an attractive component is refuted by the effect of LED lighting, which supplies negligible infrared radiation yet still entraps vast numbers of insects.\nI don\'t see why attraction to LEDs shows they\'re not seeking heat. Could they for example be evolutionarily programmed to associate light with heat? So that even though they don\'t encounter heat near/on the LEDs they still "expect" to?',
 'reasoning': 'The question probes why insects are drawn to low-heat LED lights, challenging the idea that their attraction to light is heat-based. The document helps distinguish between heat attraction and evolved behaviors, shedding light on why insects might be attracted to LEDs despite their minimal heat.',
 'id': '0',
 'excluded_ids': ['N/A'],
 'gold_ids_long': ['insects_attracted_to_light/Proximate_and_ultimate_ca

In [16]:
import weaviate

weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.getenv("WEAVIATE_CLUSTER_URL"),
    auth_credentials=weaviate.auth.AuthApiKey(api_key=os.getenv("WEAVIATE_API_KEY"))
)

collection = weaviate_client.collections.get("BrightBiology_Default")

In [41]:
from pydantic import BaseModel

class SearchResult(BaseModel):
    content: str
    dataset_id: str

def hybrid_search(collection, query, limit=10) -> list[SearchResult]:
    results = collection.query.hybrid(
        query=query,
        limit=limit,
    )    
    parsed_results = []
    
    for o in results.objects:
        parsed_results.append(SearchResult(content=o.properties["content"], dataset_id=o.properties["dataset_id"]))
    return parsed_results

hybrid_search(collection, queries[0]["query"])

[SearchResult(content=' by non-heating light sources such as color LEDs.', dataset_id='RGB_equivalent_for_smells/RGB_color_model_4_18.txt'),
 SearchResult(content='illa, light, water, chemicals (senses of taste and smell), sound, and heat. Some insects such as bees can perceive ultraviolet wavelengths, or detect polarized light, while the antennae of male moths can detect the pheromones of female moths over distances of over a kilometer. There is a trade-off between visual acuity and chemical or tactile acuity, such that most insects with well-developed eyes have reduced or simple antennae, and vice versa. Insects perceive sound by different mechanisms, such as thin vibrating membranes (tympana). Insects', dataset_id='insects_attracted_to_light/Insect_4_16.txt'),
 SearchResult(content=' cells but no lens or other means of projecting an image onto those cells. They can distinguish between light and dark but no more, enabling them to avoid direct sunlight.\nIn organisms dwelling near dee

In [22]:
def success_at_k(gold_ids: list[str], search_results: list[SearchResult], k: int) -> bool:
    """
    Check if any of the gold_ids are contained in the top K search results.
    
    Args:
        gold_ids: List of dataset IDs that should be returned (ground truth)
        search_results: List of SearchResult objects from the search
        k: Number of top results to consider
    
    Returns:
        True if at least one gold_id is found in the top k results, False otherwise
    """
    if not search_results or not gold_ids:
        return False
    
    # Get the top K results
    top_k_results = search_results[:k]
    
    # Extract dataset_ids from the top K results
    result_ids = {result.dataset_id for result in top_k_results}
    
    # Check if any gold_id is in the result_ids
    return any(gold_id in result_ids for gold_id in gold_ids)


# Example usage with a single query
query = queries[0]
results = hybrid_search(collection, query["query"])
k_values = [1, 5, 20]

print(f"Query: {query['query'][:100]}...")
print(f"Gold IDs: {query['gold_ids']}")
print(f"\nSuccess @ K:")
for k in k_values:
    success = success_at_k(query["gold_ids"], results, k)
    print(f"  Success @ {k}: {success}")

Query: Claim in article about why insects are attracted to light
In this article they are addressing the re...
Gold IDs: ['insects_attracted_to_light/Proximate_and_ultimate_causation_0.txt', 'insects_attracted_to_light/Proximate_and_ultimate_causation_1.txt', 'insects_attracted_to_light/Phototaxis_0.txt', 'insects_attracted_to_light/Phototaxis_3.txt', 'insects_attracted_to_light/Phototaxis_4.txt']

Success @ K:
  Success @ 1: False
  Success @ 5: False
  Success @ 20: False


# Hybrid Search

In [23]:
# Compute success @ K for all queries
k_values = [1, 5, 20]
success_counts = {k: 0 for k in k_values}
total_queries = len(queries)

print(f"Evaluating {total_queries} queries...")
print("=" * 50)

for i, query in enumerate(queries):
    # Run search for this query
    results = hybrid_search(collection, query["query"])
    
    # Compute success @ K for each K value
    for k in k_values:
        if success_at_k(query["gold_ids"], results, k):
            success_counts[k] += 1
    
    # Print progress every 10 queries
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{total_queries} queries...")

print(f"\nCompleted evaluation of {total_queries} queries")
print("=" * 50)
print("\nSuccess @ K Results:")
print("-" * 50)
for k in k_values:
    success_rate = success_counts[k] / total_queries
    print(f"Success @ {k:2d}: {success_counts[k]:3d}/{total_queries} ({success_rate:.1%})")

Evaluating 103 queries...
Processed 10/103 queries...
Processed 20/103 queries...
Processed 30/103 queries...
Processed 40/103 queries...
Processed 50/103 queries...
Processed 60/103 queries...
Processed 70/103 queries...
Processed 80/103 queries...
Processed 90/103 queries...
Processed 100/103 queries...

Completed evaluation of 103 queries

Success @ K Results:
--------------------------------------------------
Success @  1:  16/103 (15.5%)
Success @  5:  33/103 (32.0%)
Success @ 20:  42/103 (40.8%)


# Search Query Writer

In [27]:
import dspy

import dspy
lm = dspy.LM("openai/gpt-5-mini", api_key=os.getenv("OPENAI_API_KEY"))
dspy.configure(lm=lm)

class QueryWriter(dspy.Signature):
    """Translate the user's question into a query optimized for a search engine."""

    question: str = dspy.InputField(description="The user's question to be translated into a search query.")
    query: str = dspy.OutputField(description="The search query optimized for a search engine.")

query_writer = dspy.Predict(QueryWriter)

query_writer(question=queries[0]["query"])

Prediction(
    query='"why are insects attracted to LED lights despite negligible infrared radiation" "do insects seek heat or light" phototaxis thermotaxis "evolutionary association"'
)

In [30]:
# Compute success @ K for all queries with query writing layer
k_values = [1, 5, 20]
success_counts = {k: 0 for k in k_values}
total_queries = len(queries)

print(f"Evaluating {total_queries} queries with query writing layer...")
print("=" * 50)

for i, query in enumerate(queries):
    query_result = query_writer(question=query["query"])
    generated_query = query_result.query
    
    results = hybrid_search(collection, generated_query)
    
    for k in k_values:
        if success_at_k(query["gold_ids"], results, k):
            success_counts[k] += 1

    # Print progress and current success every 10 queries
    if (i + 1) % 10 == 0:
        progress_str = f"Processed {i + 1}/{total_queries} queries..."
        success_strs = []
        for k in k_values:
            success_rate = success_counts[k] / (i + 1)
            success_strs.append(f"Success@{k}: {success_counts[k]}/{i+1} ({success_rate:.1%})")
        print(f"{progress_str}   " + ", ".join(success_strs))

print(f"\nCompleted evaluation of {total_queries} queries")
print("=" * 50)
print("\nSuccess @ K Results (with query writing):")
print("-" * 50)
for k in k_values:
    success_rate = success_counts[k] / total_queries
    print(f"Success @ {k:2d}: {success_counts[k]:3d}/{total_queries} ({success_rate:.1%})")

Evaluating 103 queries with query writing layer...
Processed 10/103 queries...   Success@1: 6/10 (60.0%), Success@5: 9/10 (90.0%), Success@20: 9/10 (90.0%)
Processed 20/103 queries...   Success@1: 13/20 (65.0%), Success@5: 18/20 (90.0%), Success@20: 18/20 (90.0%)
Processed 30/103 queries...   Success@1: 21/30 (70.0%), Success@5: 26/30 (86.7%), Success@20: 27/30 (90.0%)
Processed 40/103 queries...   Success@1: 26/40 (65.0%), Success@5: 32/40 (80.0%), Success@20: 34/40 (85.0%)
Processed 50/103 queries...   Success@1: 28/50 (56.0%), Success@5: 38/50 (76.0%), Success@20: 40/50 (80.0%)
Processed 60/103 queries...   Success@1: 31/60 (51.7%), Success@5: 43/60 (71.7%), Success@20: 45/60 (75.0%)
Processed 70/103 queries...   Success@1: 34/70 (48.6%), Success@5: 48/70 (68.6%), Success@20: 51/70 (72.9%)
Processed 80/103 queries...   Success@1: 37/80 (46.2%), Success@5: 52/80 (65.0%), Success@20: 56/80 (70.0%)
Processed 90/103 queries...   Success@1: 39/90 (43.3%), Success@5: 56/90 (62.2%), Succes

# (TODO) Search Query Paraphrasings

# Hypothetical Document Embeddings (HyDE)

In [31]:
class HypotheticalDocumentGenerator(dspy.Signature):
    """Generate a hypothetical document that captures relevance patterns to answer the query, even if it contains factual errors or hallucinated details."""

    query: str = dspy.InputField(
        description="The search query or question that needs to be answered."
    )
    hypothetical_document: str = dspy.OutputField(
        description="A hypothetical document that demonstrates relevance patterns for the query. This document captures what a relevant answer would look like, but may contain ungrounded or hallucinated details that will be filtered by the encoder."
    )

hyde_generator = dspy.Predict(HypotheticalDocumentGenerator)

hyde_generator(query=queries[0]["query"])

Prediction(
    hypothetical_document='Summary\nMany flying insects are attracted to artificial lights mainly because of visual cues (brightness and specific short-wavelength components such as ultraviolet and blue) and because artificial point lights disrupt their natural orientation/navigation mechanisms. The observation that LEDs, which emit negligible infrared/thermal radiation compared with incandescent bulbs, still attract large numbers of insects therefore argues that heat per se is not the primary long-range cue attracting most species. That said, some insects do use thermosensation at short ranges (e.g., to find warm hosts or resting sites), and an evolutionary association between light and warmth could exist for some taxa — but such an association is unlikely to explain attraction from distances at which thermal gradients are negligible.\n\nBackground: how insects detect and respond to light\n- Phototaxis and spectral sensitivity: Many insects display positive phototaxis (an 

In [33]:
# Compute success @ K for all queries with query writing layer
k_values = [1, 5, 20]
success_counts = {k: 0 for k in k_values}
total_queries = len(queries)

print(f"Evaluating {total_queries} queries with HyDE query writing layer...")
print("=" * 50)

for i, query in enumerate(queries):
    query_result = hyde_generator(query=query["query"])
    generated_query = query_result.hypothetical_document
    
    results = hybrid_search(collection, generated_query)
    
    for k in k_values:
        if success_at_k(query["gold_ids"], results, k):
            success_counts[k] += 1

    # Print progress and current success every 10 queries
    if (i + 1) % 10 == 0:
        progress_str = f"Processed {i + 1}/{total_queries} queries..."
        success_strs = []
        for k in k_values:
            success_rate = success_counts[k] / (i + 1)
            success_strs.append(f"Success@{k}: {success_counts[k]}/{i+1} ({success_rate:.1%})")
        print(f"{progress_str}   " + ", ".join(success_strs))

print(f"\nCompleted evaluation of {total_queries} queries")
print("=" * 50)
print("\nSuccess @ K Results (with HyDE query writing):")
print("-" * 50)
for k in k_values:
    success_rate = success_counts[k] / total_queries
    print(f"Success @ {k:2d}: {success_counts[k]:3d}/{total_queries} ({success_rate:.1%})")

Evaluating 103 queries with HyDE query writing layer...
Processed 10/103 queries...   Success@1: 7/10 (70.0%), Success@5: 9/10 (90.0%), Success@20: 9/10 (90.0%)
Processed 20/103 queries...   Success@1: 16/20 (80.0%), Success@5: 18/20 (90.0%), Success@20: 19/20 (95.0%)
Processed 30/103 queries...   Success@1: 23/30 (76.7%), Success@5: 26/30 (86.7%), Success@20: 28/30 (93.3%)
Processed 40/103 queries...   Success@1: 30/40 (75.0%), Success@5: 35/40 (87.5%), Success@20: 37/40 (92.5%)
Processed 50/103 queries...   Success@1: 31/50 (62.0%), Success@5: 40/50 (80.0%), Success@20: 44/50 (88.0%)
Processed 60/103 queries...   Success@1: 35/60 (58.3%), Success@5: 46/60 (76.7%), Success@20: 50/60 (83.3%)
Processed 70/103 queries...   Success@1: 38/70 (54.3%), Success@5: 52/70 (74.3%), Success@20: 56/70 (80.0%)
Processed 80/103 queries...   Success@1: 41/80 (51.2%), Success@5: 59/80 (73.8%), Success@20: 63/80 (78.8%)
Processed 90/103 queries...   Success@1: 45/90 (50.0%), Success@5: 65/90 (72.2%), S

# Pseudo-Relevance Feedback (PRF)

In [37]:
class QueryRefiner(dspy.Signature):
    """Refine the original query based on initial search results to improve retrieval in subsequent searches."""

    original_query: str = dspy.InputField(
        description="The user's original search query or question."
    )
    search_results: str = dspy.InputField(
        description="Initial search results (titles, snippets, or passages) that provide context about what the corpus contains and how it's structured."
    )
    refined_query: str = dspy.OutputField(
        description="An improved search query that incorporates insights from the initial results, such as better terminology, specific entities, or domain-specific language found in the corpus."
    )

def search_results_to_str(search_results: list[SearchResult]) -> str:
    return "\n".join([f"{i+1}. {result.content}" for i, result in enumerate(search_results)])

# Usage example
query_refiner = dspy.Predict(QueryRefiner)

# Example with search results
initial_results = hybrid_search(collection, queries[0]["query"])
initial_results_str = search_results_to_str(initial_results)

query_refiner(original_query=queries[0]["query"], search_results=initial_results_str).refined_query

'Why are insects attracted to artificial light despite LEDs emitting negligible infrared? Compare heat-radiation (thermotaxis) vs phototaxis hypotheses; evidence from LED vs incandescent experiments, spectral sensitivity (UV/blue), thermoreceptors in insects, evolutionary association between light and heat, and navigational (celestial/transverse orientation) explanations for "flight-to-light" behavior in moths and other nocturnal insects'

In [42]:
# Measure and compare success@K for original and PRF-refined queries;
# also track exclusive gold hits

k_values = [1, 5, 20]
total_queries = len(queries)

# Success counters for original and PRF queries
success_counts_orig = {k: 0 for k in k_values}
success_counts_prf = {k: 0 for k in k_values}

# Track exclusive hits for analysis
orig_only_hits = {k: 0 for k in k_values}    # gold doc found by original but not PRF
prf_only_hits = {k: 0 for k in k_values}     # gold doc found by PRF but not original

print(f"Evaluating {total_queries} queries: original vs PRF query writing layer...")
print("=" * 50)

for i, query in enumerate(queries):
    # Initial search with the original query
    initial_results = hybrid_search(collection, query["query"], limit=5)
    initial_results_str = search_results_to_str(initial_results)

    # Refine the query with PRF/LLM
    query_result = query_refiner(original_query=query["query"], search_results=initial_results_str)
    generated_query = query_result.refined_query

    # Search using the refined (PRF) query
    prf_results = hybrid_search(collection, generated_query)

    for k in k_values:
        gold_ids = query["gold_ids"]
        orig_success = success_at_k(gold_ids, initial_results, k)
        prf_success = success_at_k(gold_ids, prf_results, k)

        if orig_success:
            success_counts_orig[k] += 1
        if prf_success:
            success_counts_prf[k] += 1

        # Exclusive counts
        if orig_success and not prf_success:
            orig_only_hits[k] += 1
        if prf_success and not orig_success:
            prf_only_hits[k] += 1

    # Print progress every 10 queries
    if (i + 1) % 10 == 0:
        progress_str = f"Processed {i + 1}/{total_queries} queries..."
        lines = []
        for k in k_values:
            orig_rate = success_counts_orig[k] / (i + 1)
            prf_rate = success_counts_prf[k] / (i + 1)
            orig_only = orig_only_hits[k]
            prf_only = prf_only_hits[k]
            line = (f"K={k:2d} | Orig: {success_counts_orig[k]}/{i+1} ({orig_rate:.1%})"
                    f" | PRF: {success_counts_prf[k]}/{i+1} ({prf_rate:.1%})"
                    f" | Orig-only: {orig_only}  PRF-only: {prf_only}")
            lines.append(line)
        print(f"{progress_str}\n  " + "\n  ".join(lines))

print(f"\nCompleted evaluation of {total_queries} queries")
print("=" * 50)
print("\nSuccess @ K Results (Original vs PRF query writing):")
print("-" * 50)
for k in k_values:
    orig_rate = success_counts_orig[k] / total_queries
    prf_rate = success_counts_prf[k] / total_queries
    orig_only = orig_only_hits[k]
    prf_only = prf_only_hits[k]
    print(
        f"Success @ {k:2d} | Orig: {success_counts_orig[k]:3d}/{total_queries} ({orig_rate:.1%})"
        f" | PRF: {success_counts_prf[k]:3d}/{total_queries} ({prf_rate:.1%})"
        f" | Orig-only: {orig_only:3d}  PRF-only: {prf_only:3d}"
    )

Evaluating 103 queries: original vs PRF query writing layer...
Processed 10/103 queries...
  K= 1 | Orig: 2/10 (20.0%) | PRF: 8/10 (80.0%) | Orig-only: 0  PRF-only: 6
  K= 5 | Orig: 5/10 (50.0%) | PRF: 9/10 (90.0%) | Orig-only: 0  PRF-only: 4
  K=20 | Orig: 5/10 (50.0%) | PRF: 9/10 (90.0%) | Orig-only: 0  PRF-only: 4
Processed 20/103 queries...
  K= 1 | Orig: 5/20 (25.0%) | PRF: 15/20 (75.0%) | Orig-only: 0  PRF-only: 10
  K= 5 | Orig: 11/20 (55.0%) | PRF: 18/20 (90.0%) | Orig-only: 0  PRF-only: 7
  K=20 | Orig: 11/20 (55.0%) | PRF: 18/20 (90.0%) | Orig-only: 0  PRF-only: 7
Processed 30/103 queries...
  K= 1 | Orig: 7/30 (23.3%) | PRF: 23/30 (76.7%) | Orig-only: 0  PRF-only: 16
  K= 5 | Orig: 17/30 (56.7%) | PRF: 27/30 (90.0%) | Orig-only: 0  PRF-only: 10
  K=20 | Orig: 17/30 (56.7%) | PRF: 27/30 (90.0%) | Orig-only: 0  PRF-only: 10
Processed 40/103 queries...
  K= 1 | Orig: 8/40 (20.0%) | PRF: 30/40 (75.0%) | Orig-only: 0  PRF-only: 22
  K= 5 | Orig: 18/40 (45.0%) | PRF: 35/40 (87.5%)

# Multiple Search Query Writer

In [ ]:
class MultiQueryGenerator(dspy.Signature):
    """Generate multiple diverse search queries that approach the information need from different angles."""

    original_query: str = dspy.InputField(
        description="The user's original search query or question."
    )
    num_queries: int = dspy.InputField(
        description="Number of diverse queries to generate (typically 5-10).",
    )
    queries: list[str] = dspy.OutputField(
        description="A list of diverse search queries that rephrase, expand, or reframe the original query using different terminology, specificity levels, and perspectives. Each query should target the same information need but use varied language to maximize retrieval coverage."
    )

multi_query_gen = dspy.Predict(MultiQueryGenerator)

result = multi_query_gen(
    original_query=queries[0]["query"],
    num_queries=5
)

for query in result.queries:
    print(query)
    print("-" * 50)

Do insects go to lights because they seek heat? Evidence comparing attraction to LED (low IR) versus incandescent (high IR) bulbs
--------------------------------------------------
Could insects be evolutionarily or innately programmed to associate visible light with warmth, causing attraction to LEDs even when they emit negligible heat?
--------------------------------------------------
Mechanisms of insect attraction to artificial light: phototaxis, transverse orientation, thermal cues, and why LEDs still trap insects
--------------------------------------------------
Research papers and experiments showing LEDs attract insects despite little infrared radiation — proposed explanations and behavioral tests
--------------------------------------------------
Have conditioning or learning experiments shown that moths/flies can associate light with heat, or is light attraction purely innate sensory navigation?
--------------------------------------------------


In [47]:
# Measure and compare success@K for original and multi-query strategies;
# also track exclusive gold hits

k_values = [1, 5, 20]
total_queries = len(queries)

# Success counters for original and multi-query strategies
success_counts_orig = {k: 0 for k in k_values}
success_counts_multi = {k: 0 for k in k_values}

# Track exclusive hits for analysis
orig_only_hits = {k: 0 for k in k_values}    # gold doc found by original but not multi-query
multi_only_hits = {k: 0 for k in k_values}   # gold doc found by multi-query but not original

# Number of queries to generate per original query
num_generated_queries = 5

print(f"Evaluating {total_queries} queries: original vs multi-query generation...")
print(f"Generating {num_generated_queries} queries per original query")
print("=" * 50)

for i, query in enumerate(queries):
    # Initial search with the original query
    initial_results = hybrid_search(collection, query["query"])

    # Generate multiple queries
    multi_query_result = multi_query_gen(
        original_query=query["query"],
        num_queries=num_generated_queries
    )
    generated_queries = multi_query_result.queries

    # Collect all results from generated queries
    # We'll aggregate results and check if ANY of them contain the gold document
    all_multi_results = []
    for gen_query in generated_queries:
        gen_results = hybrid_search(collection, gen_query)
        all_multi_results.extend(gen_results)
    
    # Deduplicate results by document ID (keeping first occurrence)
    seen_ids = set()
    deduped_multi_results = []
    for result in all_multi_results:
        doc_id = result.dataset_id
        if doc_id not in seen_ids:
            seen_ids.add(doc_id)
            deduped_multi_results.append(result)

    for k in k_values:
        gold_ids = query["gold_ids"]
        orig_success = success_at_k(gold_ids, initial_results, k)
        # For multi-query, we check top-k from the aggregated results
        multi_success = success_at_k(gold_ids, deduped_multi_results, k)

        if orig_success:
            success_counts_orig[k] += 1
        if multi_success:
            success_counts_multi[k] += 1

        # Exclusive counts
        if orig_success and not multi_success:
            orig_only_hits[k] += 1
        if multi_success and not orig_success:
            multi_only_hits[k] += 1

    # Print progress every 10 queries
    if (i + 1) % 10 == 0:
        progress_str = f"Processed {i + 1}/{total_queries} queries..."
        lines = []
        for k in k_values:
            orig_rate = success_counts_orig[k] / (i + 1)
            multi_rate = success_counts_multi[k] / (i + 1)
            orig_only = orig_only_hits[k]
            multi_only = multi_only_hits[k]
            line = (f"K={k:2d} | Orig: {success_counts_orig[k]}/{i+1} ({orig_rate:.1%})"
                    f" | Multi: {success_counts_multi[k]}/{i+1} ({multi_rate:.1%})"
                    f" | Orig-only: {orig_only}  Multi-only: {multi_only}")
            lines.append(line)
        print(f"{progress_str}\n  " + "\n  ".join(lines))

print(f"\nCompleted evaluation of {total_queries} queries")
print("=" * 50)
print("\nSuccess @ K Results (Original vs Multi-Query Generation):")
print("-" * 50)
for k in k_values:
    orig_rate = success_counts_orig[k] / total_queries
    multi_rate = success_counts_multi[k] / total_queries
    orig_only = orig_only_hits[k]
    multi_only = multi_only_hits[k]
    improvement = multi_rate - orig_rate
    print(
        f"Success @ {k:2d} | Orig: {success_counts_orig[k]:3d}/{total_queries} ({orig_rate:.1%})"
        f" | Multi: {success_counts_multi[k]:3d}/{total_queries} ({multi_rate:.1%})"
        f" | Δ: {improvement:+.1%}"
        f" | Orig-only: {orig_only:3d}  Multi-only: {multi_only:3d}"
    )

Evaluating 103 queries: original vs multi-query generation...
Generating 5 queries per original query
Processed 10/103 queries...
  K= 1 | Orig: 2/10 (20.0%) | Multi: 4/10 (40.0%) | Orig-only: 0  Multi-only: 2
  K= 5 | Orig: 5/10 (50.0%) | Multi: 7/10 (70.0%) | Orig-only: 0  Multi-only: 2
  K=20 | Orig: 6/10 (60.0%) | Multi: 10/10 (100.0%) | Orig-only: 0  Multi-only: 4
Processed 20/103 queries...
  K= 1 | Orig: 5/20 (25.0%) | Multi: 6/20 (30.0%) | Orig-only: 1  Multi-only: 2
  K= 5 | Orig: 11/20 (55.0%) | Multi: 13/20 (65.0%) | Orig-only: 2  Multi-only: 4
  K=20 | Orig: 12/20 (60.0%) | Multi: 20/20 (100.0%) | Orig-only: 0  Multi-only: 8
Processed 30/103 queries...
  K= 1 | Orig: 7/30 (23.3%) | Multi: 11/30 (36.7%) | Orig-only: 1  Multi-only: 5
  K= 5 | Orig: 17/30 (56.7%) | Multi: 20/30 (66.7%) | Orig-only: 2  Multi-only: 5
  K=20 | Orig: 20/30 (66.7%) | Multi: 30/30 (100.0%) | Orig-only: 0  Multi-only: 10
Processed 40/103 queries...
  K= 1 | Orig: 8/40 (20.0%) | Multi: 16/40 (40.0%) |